# Nematode classifiers

Train AlexNet, VGG16, and ResNet-50 on dataset_inema/train. Validate only on dataset_inema/val.

In [ ]:
%pip install -q timm pillow pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from pathlib import Path

ROOT = Path('/home/boibhai/GitHub/nematode-detection')
TRAIN_DIR = ROOT / 'dataset_inema' / 'train'
VAL_DIR = ROOT / 'dataset_inema' / 'val'
PRETRAINED_DIR = ROOT / 'pretrained_models'
TRAINED_DIR = ROOT / 'trained_models'
PRETRAINED_DIR.mkdir(exist_ok=True)
TRAINED_DIR.mkdir(exist_ok=True)

# torchvision and timm store downloaded pretrained weights here, not in a temporary kernel cache.
os.environ['TORCH_HOME'] = str(PRETRAINED_DIR)
os.environ['HF_HOME'] = str(PRETRAINED_DIR / 'huggingface')

import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = 224
BATCH_SIZE = 2       # VGG16 fits safely on this 3.68 GB GPU at this batch size.
EPOCHS = 10
LR = 1e-4
print('Device:', DEVICE)

/home/boibhai/.virtualenvs/boidefault/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/boibhai/.virtualenvs/boidefault/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((.485, .456, .406), (.229, .224, .225)),
])

train_set = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_set = datasets.ImageFolder(VAL_DIR, transform=transform)
assert train_set.classes == val_set.classes, 'Train and validation class folders must match.'
classes = train_set.classes

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_test_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(len(classes), 'classes |', len(train_set), 'train images |', len(val_set), 'validation images')

19 classes | 4434 train images | 552 validation images


In [10]:
def clear_gpu():
    # Releases models left in notebook variables between separate training runs.
    for value in list(globals().values()):
        if isinstance(value, nn.Module):
            value.to('cpu')
    gc.collect()
    torch.cuda.empty_cache()

def make_alexnet():
    model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(classes))
    return model

def make_vgg16():
    return timm.create_model('vgg16.tv_in1k', pretrained=True, num_classes=len(classes))

def make_resnet50():
    return timm.create_model('resnet50', pretrained=True, num_classes=len(classes))

def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for images, labels in loader:
            predictions = model(images.to(DEVICE)).argmax(1).cpu()
            correct += (predictions == labels).sum().item()
            total += len(labels)
    return correct / total

def train(model, name):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    best_accuracy = 0
    best_weights = None

    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = loss_fn(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        validation_accuracy = accuracy(model, val_loader)
        print(f'{name} epoch {epoch + 1}/{EPOCHS}: validation accuracy = {validation_accuracy:.3f}')
        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_weights)
    return model, best_accuracy

def save_model(model, name):
    path = TRAINED_DIR / f'{name}_nematode.pth'
    torch.save({'classes': classes, 'state_dict': model.state_dict()}, path)
    print('Saved:', path)

In [14]:
# AlexNet: run this cell independently.
clear_gpu()
alexnet, alexnet_val = train(make_alexnet(), 'AlexNet')
alexnet_train = accuracy(alexnet, train_test_loader)
save_model(alexnet, 'alexnet')
print(f'AlexNet train={alexnet_train:.3f}, validation={alexnet_val:.3f}')
alexnet.to('cpu')
del alexnet
torch.cuda.empty_cache()

AlexNet epoch 1/10: validation accuracy = 0.433
AlexNet epoch 2/10: validation accuracy = 0.426
AlexNet epoch 3/10: validation accuracy = 0.458
AlexNet epoch 4/10: validation accuracy = 0.531
AlexNet epoch 5/10: validation accuracy = 0.422
AlexNet epoch 6/10: validation accuracy = 0.489
AlexNet epoch 7/10: validation accuracy = 0.525
AlexNet epoch 8/10: validation accuracy = 0.507
AlexNet epoch 9/10: validation accuracy = 0.493
AlexNet epoch 10/10: validation accuracy = 0.533
Saved: /home/boibhai/GitHub/nematode-detection/trained_models/alexnet_nematode.pth
AlexNet train=0.964, validation=0.533


In [11]:
# 1. Load the checkpoint dictionary using your prewritten DEVICE
checkpoint = torch.load('trained_models/alexnet_nematode.pth', map_location=DEVICE, weights_only=False)

# 2. Extract the class names and weights dictionary
classes = checkpoint['classes']
state_dict = checkpoint['state_dict']

print("Loaded classes:", classes)

# 3. Initialize AlexNet and replace the final linear layer (index 6)
alexnet = models.alexnet(weights=None)
alexnet.classifier[6] = nn.Linear(in_features=4096, out_features=len(classes))

# 4. Load the extracted state_dict into the model
alexnet.load_state_dict(state_dict)

# 5. Move to DEVICE and set to evaluation mode
alexnet = alexnet.to(DEVICE)
alexnet.eval()

Loaded classes: ['Acrobeles', 'Acrobeloides', 'Amplimerlinius', 'Aphelenchoides', 'Aporcelaimus', 'Axonchium', 'Discolimus', 'Ditylenchus', 'Dorylaimus', 'Eudorylaimus', 'Helicotylenchus', 'Mesodorylaimus', 'Miconchus', 'Mylonchulus', 'Panagrolaimus', 'Pratylenchus', 'Pristionchus', 'Rhbiditis', 'Xenocriconema']


AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [ ]:
# training already trained alexnet

IMAGE_SIZE = 224
BATCH_SIZE = 2       # VGG16 fits safely on this 3.68 GB GPU at this batch size.
EPOCHS = 10
LR = 1e-4

clear_gpu()
alexnet, alexnet_val = train(alexnet, 'AlexNet')
alexnet_train = accuracy(alexnet, train_test_loader)
save_model(alexnet, 'alexnet')
print(f'AlexNet train={alexnet_train:.3f}, validation={alexnet_val:.3f}')
alexnet.to('cpu')
del alexnet
torch.cuda.empty_cache()